In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from ultralytics import YOLO

In [ ]:
BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

YOLO_DIR = BASE_DIR / "yolo" / "oral_seg_yolo"
YOLO_YAML = YOLO_DIR / "oral_seg_yolo.yaml"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"

YOLO_RUNS_DIR = OUTPUT_DIR / "yolo_runs"

CSV_YOLO = TABLES_DIR / "oral_seg_yolo_dataset.csv"

for carpeta in [TABLES_DIR, FIGURES_DIR, MODELS_DIR, YOLO_RUNS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("YOLO_YAML existe:", YOLO_YAML.exists())
print("CSV_YOLO existe:", CSV_YOLO.exists())
print("YOLO_DIR:", YOLO_DIR)

In [ ]:
df = pd.read_csv(CSV_YOLO)

print("Total muestras:", len(df))

print("\nSplit:")
print(df["split"].value_counts())

print("\nOrigen:")
print(df["origen"].value_counts())

print("\nClase lengua:")
print(df["clase_lengua"].value_counts())

display(df.head())

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 20

RUN_NAME = "oral_seg_yolo11n_seg_light_compare"

print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("EPOCHS:", EPOCHS)
print("RUN_NAME:", RUN_NAME)

In [ ]:
model = YOLO("yolo11n-seg.pt")

In [ ]:
results = model.train(
    data=str(YOLO_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(YOLO_RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True
)

In [ ]:
best_model_path = YOLO_RUNS_DIR / RUN_NAME / "weights" / "best.pt"

print("Best model existe:", best_model_path.exists())
print("Best model path:", best_model_path)

In [ ]:
best_model = YOLO(str(best_model_path))

metrics = best_model.val(
    data=str(YOLO_YAML),
    split="test",
    imgsz=IMG_SIZE,
    device=DEVICE
)

### Evaluación manual corregida

En una primera evaluación, Dice salía mayor que 1 debido a que algunas máscaras reales se cargaban con forma `(256, 256, 1)` mientras que las predicciones YOLO estaban en `(256, 256)`. Para evitar broadcasting incorrecto en NumPy, se fuerza cada máscara a 2D antes de calcular Dice e IoU.

In [ ]:
df_test = df[df["split"] == "test"].copy().reset_index(drop=True)


def asegurar_2d(mask):
    """
    Fuerza una máscara a 2D para evitar broadcasting raro.
    """
    mask = np.array(mask)
    
    mask = np.squeeze(mask)
    
    if mask.ndim == 3:
        mask = mask[:, :, 0]
    
    return mask


def prediccion_yolo_a_mascara_v3(resultado, alto, ancho):
    """
    Convierte YOLO segmentation a máscara binaria 2D.
    """
    mask_final = np.zeros((alto, ancho), dtype=np.uint8)
    
    if resultado.masks is None:
        return mask_final
    
    masks = resultado.masks.data.cpu().numpy()
    
    if masks is None or len(masks) == 0:
        return mask_final
    
    mask_union = np.max(masks, axis=0)
    mask_union = asegurar_2d(mask_union)
    
    mask_union = (mask_union > 0.5).astype(np.uint8)
    
    mask_union = cv2.resize(
        mask_union,
        (ancho, alto),
        interpolation=cv2.INTER_NEAREST
    )
    
    mask_union = asegurar_2d(mask_union)
    mask_union = (mask_union > 0).astype(np.uint8)
    
    return mask_union


def dice_seguro_v3(mask_real, mask_pred):
    real = asegurar_2d(mask_real)
    pred = asegurar_2d(mask_pred)
    
    real = (real > 0).astype(np.float32)
    pred = (pred > 0).astype(np.float32)
    
    if real.shape != pred.shape:
        print("Shape real:", real.shape)
        print("Shape pred:", pred.shape)
        raise ValueError("Las máscaras no tienen la misma forma")
    
    inter = float(np.sum(real * pred))
    total = float(np.sum(real) + np.sum(pred))
    
    if total == 0:
        return 1.0
    
    return (2.0 * inter) / total


def iou_seguro_v3(mask_real, mask_pred):
    real = asegurar_2d(mask_real)
    pred = asegurar_2d(mask_pred)
    
    real = real > 0
    pred = pred > 0
    
    if real.shape != pred.shape:
        print("Shape real:", real.shape)
        print("Shape pred:", pred.shape)
        raise ValueError("Las máscaras no tienen la misma forma")
    
    inter = float(np.logical_and(real, pred).sum())
    union = float(np.logical_or(real, pred).sum())
    
    if union == 0:
        return 1.0
    
    return inter / union


filas_metricas = []

for i, fila in df_test.iterrows():
    img_path = fila["img_yolo"]
    mask_path = fila["mask_final"]
    
    mask_real = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    if mask_real is None:
        continue
    
    mask_real = asegurar_2d(mask_real)
    
    alto, ancho = mask_real.shape[:2]
    
    pred = best_model.predict(
        source=img_path,
        imgsz=IMG_SIZE,
        device=DEVICE,
        conf=0.25,
        verbose=False
    )
    
    mask_pred = prediccion_yolo_a_mascara_v3(pred[0], alto, ancho)
    
    d = dice_seguro_v3(mask_real, mask_pred)
    j = iou_seguro_v3(mask_real, mask_pred)
    
    filas_metricas.append({
        "id_global": fila["id_global"],
        "origen": fila["origen"],
        "clase_lengua": fila["clase_lengua"],
        "dice": d,
        "iou": j,
        "filename_original": fila["filename_original"],
        "img_yolo": img_path,
        "mask_final": mask_path
    })

df_metricas = pd.DataFrame(filas_metricas)

print("Métricas calculadas:", len(df_metricas))
display(df_metricas.head())

print("Comprobación de rangos:")
print("Dice min:", df_metricas["dice"].min())
print("Dice max:", df_metricas["dice"].max())
print("IoU min:", df_metricas["iou"].min())
print("IoU max:", df_metricas["iou"].max())

assert df_metricas["dice"].max() <= 1.0, "ERROR: Dice sigue siendo mayor que 1"
assert df_metricas["iou"].max() <= 1.0, "ERROR: IoU sigue siendo mayor que 1"

In [ ]:
print("Media global:")
display(df_metricas[["dice", "iou"]].mean())

print("Por origen:")
display(df_metricas.groupby("origen")[["dice", "iou"]].mean())

print("Por clase:")
display(df_metricas.groupby("clase_lengua")[["dice", "iou"]].mean())

print("Por origen y clase:")
display(df_metricas.groupby(["origen", "clase_lengua"])[["dice", "iou"]].mean())


csv_metricas = TABLES_DIR / "oral_seg_yolo_light_metricas_por_imagen.csv"
df_metricas.to_csv(csv_metricas, index=False)

df_resumen = pd.DataFrame([{
    "modelo": "YOLO11n-seg",
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "test_dice": df_metricas["dice"].mean(),
    "test_iou": df_metricas["iou"].mean()
}])

csv_resumen = TABLES_DIR / "oral_seg_yolo_light_resultados_test.csv"
df_resumen.to_csv(csv_resumen, index=False)

display(df_resumen)

print("Métricas por imagen guardadas en:", csv_metricas)
print("Resumen YOLO guardado en:", csv_resumen)

In [ ]:
muestras = df_test.sample(
    n=min(8, len(df_test)),
    random_state=7
).reset_index(drop=True)

plt.figure(figsize=(12, 12))

for i, fila in muestras.iterrows():
    img_path = fila["img_yolo"]
    mask_path = fila["mask_final"]
    
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    mask_real = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    if img is None or mask_real is None:
        continue
    
    mask_real = asegurar_2d(mask_real)
    alto, ancho = mask_real.shape[:2]
    
    pred = best_model.predict(
        source=img_path,
        imgsz=IMG_SIZE,
        device=DEVICE,
        conf=0.25,
        verbose=False
    )
    
    mask_pred = prediccion_yolo_a_mascara_v3(pred[0], alto, ancho)
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(4, 4, 2*i + 1)
    plt.imshow(img_rgb)
    plt.imshow(mask_real, cmap="jet", alpha=0.35)
    plt.title("Real " + fila["clase_lengua"], fontsize=8)
    plt.axis("off")
    
    plt.subplot(4, 4, 2*i + 2)
    plt.imshow(img_rgb)
    plt.imshow(mask_pred, cmap="jet", alpha=0.35)
    plt.title("YOLO pred", fontsize=8)
    plt.axis("off")

plt.tight_layout()

fig_path = FIGURES_DIR / "oral_seg_yolo_light_predicciones_test.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)

In [ ]:
csv_unet_metricas = TABLES_DIR / "oral_seg_unet_light_metricas_por_imagen.csv"
csv_yolo_metricas = TABLES_DIR / "oral_seg_yolo_light_metricas_por_imagen.csv"

df_unet_m = pd.read_csv(csv_unet_metricas)
df_yolo_m = pd.read_csv(csv_yolo_metricas)

df_comp = pd.merge(
    df_unet_m,
    df_yolo_m,
    on="id_global",
    suffixes=("_unet", "_yolo")
)

print("Imágenes comunes:", len(df_comp))

tabla_comp = pd.DataFrame([{
    "modelo": "U-Net light",
    "test_dice": df_comp["dice_unet"].mean(),
    "test_iou": df_comp["iou_unet"].mean()
}, {
    "modelo": "YOLO11n-seg",
    "test_dice": df_comp["dice_yolo"].mean(),
    "test_iou": df_comp["iou_yolo"].mean()
}])

display(tabla_comp)

csv_comp = TABLES_DIR / "oral_seg_comparativa_unet_yolo_comun.csv"
tabla_comp.to_csv(csv_comp, index=False)

print("Comparativa común guardada en:", csv_comp)

In [ ]:
# Comparación por origen

resumen_unet_origen = df_comp.groupby("origen_unet")[["dice_unet", "iou_unet"]].mean().reset_index()
resumen_yolo_origen = df_comp.groupby("origen_yolo")[["dice_yolo", "iou_yolo"]].mean().reset_index()

resumen_unet_origen.columns = ["origen", "dice", "iou"]
resumen_yolo_origen.columns = ["origen", "dice", "iou"]

resumen_unet_origen["modelo"] = "U-Net light"
resumen_yolo_origen["modelo"] = "YOLO11n-seg"

resumen_origen = pd.concat(
    [resumen_unet_origen, resumen_yolo_origen],
    ignore_index=True
)

display(resumen_origen)

csv_origen = TABLES_DIR / "oral_seg_comparativa_por_origen.csv"
resumen_origen.to_csv(csv_origen, index=False)

print("Comparativa por origen guardada en:", csv_origen)


# Comparación por clase

resumen_unet_clase = df_comp.groupby("clase_lengua_unet")[["dice_unet", "iou_unet"]].mean().reset_index()
resumen_yolo_clase = df_comp.groupby("clase_lengua_yolo")[["dice_yolo", "iou_yolo"]].mean().reset_index()

resumen_unet_clase.columns = ["clase_lengua", "dice", "iou"]
resumen_yolo_clase.columns = ["clase_lengua", "dice", "iou"]

resumen_unet_clase["modelo"] = "U-Net light"
resumen_yolo_clase["modelo"] = "YOLO11n-seg"

resumen_clase = pd.concat(
    [resumen_unet_clase, resumen_yolo_clase],
    ignore_index=True
)

display(resumen_clase)

csv_clase = TABLES_DIR / "oral_seg_comparativa_por_clase.csv"
resumen_clase.to_csv(csv_clase, index=False)

print("Comparativa por clase guardada en:", csv_clase)